In [1]:
!pip install numpy==1.26.4 --quiet
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install networkx sentence-transformers --quiet
!pip install torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.3 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# Download SNAP Facebook dataset directly

import os
import zipfile

# ─── Download dataset ───
!wget -q https://snap.stanford.edu/data/facebook_large.zip

# ─── Extract to writable directory ───
extract_path = "/kaggle/working/facebook_data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile("facebook_large.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset ready at:", extract_path)

# ─── Show files ───
print(os.listdir(extract_path))

✅ Dataset ready at: /kaggle/working/facebook_data
['facebook_large']


In [5]:
import os

DATA_PATH = "/kaggle/working/facebook_data/facebook_large"

edges_path = os.path.join(DATA_PATH, "musae_facebook_edges.csv")
features_path = os.path.join(DATA_PATH, "musae_facebook_features.json")
target_path = os.path.join(DATA_PATH, "musae_facebook_target.csv")

print(edges_path)
print(features_path)
print(target_path)

/kaggle/working/facebook_data/facebook_large/musae_facebook_edges.csv
/kaggle/working/facebook_data/facebook_large/musae_facebook_features.json
/kaggle/working/facebook_data/facebook_large/musae_facebook_target.csv


In [6]:
import csv

edge_index = []

with open(edges_path, "r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    
    for row in reader:
        src = int(row[0])
        dst = int(row[1])
        
        edge_index.append([src, dst])
        edge_index.append([dst, src])  # undirected

print("Edges loaded:", len(edge_index))

Edges loaded: 342004


In [7]:
import json
import numpy as np

with open(features_path, "r") as f:
    features_dict = json.load(f)

# Get all nodes
nodes = sorted([int(k) for k in features_dict.keys()])
node_to_idx = {node: i for i, node in enumerate(nodes)}

# Build feature matrix
max_feature_index = 0
for feats in features_dict.values():
    if len(feats) > 0:
        max_feature_index = max(max_feature_index, max(feats))

num_features = max_feature_index + 1

X = np.zeros((len(nodes), num_features))

for node, feats in features_dict.items():
    idx = node_to_idx[int(node)]
    X[idx, feats] = 1

print("Feature matrix shape:", X.shape)

Feature matrix shape: (22470, 4714)


In [8]:
import csv
from sklearn.preprocessing import LabelEncoder

labels_dict = {}

with open(target_path, "r") as f:
    reader = csv.DictReader(f)
    
    print("Columns:", reader.fieldnames)  # 🔍 DEBUG
    
    for row in reader:
        node = int(row["id"])
        
        # ✅ CORRECT LABEL COLUMN
        label = row["page_type"]
        
        labels_dict[node] = label

# Build label list
all_labels = [labels_dict[node] for node in nodes]

# Encode labels
le = LabelEncoder()
Y = le.fit_transform(all_labels)

print("Total classes:", len(set(Y)))

Columns: ['id', 'facebook_id', 'page_name', 'page_type']
Total classes: 4


In [9]:
from collections import Counter

counts = Counter(Y)
print(counts.most_common(10))

[(1, 6880), (0, 6495), (2, 5768), (3, 3327)]


In [12]:
import torch

edge_list = []

with open(edges_path, "r") as f:
    import csv
    reader = csv.reader(f)
    next(reader)  # skip header
    
    for row in reader:
        src = int(row[0])
        dst = int(row[1])
        
        src_idx = node_to_idx[src]
        dst_idx = node_to_idx[dst]
        
        edge_list.append([src_idx, dst_idx])
        edge_list.append([dst_idx, src_idx])  # undirected

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print("Edge index shape:", edge_index.shape)

Edge index shape: torch.Size([2, 342004])


In [13]:
import torch

edge_list = []

with open(edges_path, "r") as f:
    import csv
    reader = csv.reader(f)
    next(reader)  # skip header
    
    for row in reader:
        src = int(row[0])
        dst = int(row[1])
        
        src_idx = node_to_idx[src]
        dst_idx = node_to_idx[dst]
        
        edge_list.append([src_idx, dst_idx])
        edge_list.append([dst_idx, src_idx])  # undirected

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print("Edge index shape:", edge_index.shape)

Edge index shape: torch.Size([2, 342004])


In [14]:
from torch_geometric.data import Data

x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(Y, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)
data = data.to(device)
print(data)

Data(x=[22470, 4714], edge_index=[2, 342004], y=[22470])


In [15]:
from collections import Counter
from sklearn.model_selection import train_test_split
import numpy as np
import torch

# Count class frequencies
counts = Counter(Y)

# Keep only classes with >= 2 samples
valid_classes = {cls for cls, cnt in counts.items() if cnt >= 2}

# Filter indices
valid_indices = [i for i, label in enumerate(Y) if label in valid_classes]

# Safety check (IMPORTANT)
if len(valid_indices) == 0:
    raise ValueError("❌ No valid classes found. Check your label extraction (Y is likely wrong).")

# Filter labels for stratification
Y_filtered = np.array([Y[i] for i in valid_indices])

# If too few samples for stratification, fallback to random split
if len(set(Y_filtered)) < 2:
    print("⚠️ Not enough classes for stratified split. Using random split instead.")
    train_idx, test_idx = train_test_split(
        valid_indices,
        test_size=0.2,
        random_state=42
    )
else:
    # Stratified split
    train_idx, test_idx = train_test_split(
        valid_indices,
        test_size=0.2,
        stratify=Y_filtered,
        random_state=42
    )

# Create masks
train_mask = torch.zeros(len(Y), dtype=torch.bool)
test_mask = torch.zeros(len(Y), dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.test_mask = test_mask

print("Train size:", train_mask.sum().item())
print("Test size:", test_mask.sum().item())
print("Total nodes used:", len(valid_indices))
print("Classes used:", len(set(Y_filtered)))

Train size: 17976
Test size: 4494
Total nodes used: 22470
Classes used: 4


In [16]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

model = GraphSAGE(data.num_features, 64, len(set(Y))).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [20]:
import torch.nn.functional as F

batch_size = 1024
train_indices = data.train_mask.nonzero(as_tuple=True)[0]

def train():
    model.train()
    total_loss = 0

    # Shuffle training nodes
    perm = train_indices[torch.randperm(len(train_indices))]

    for i in range(0, len(perm), batch_size):
        batch_nodes = perm[i:i + batch_size]

        optimizer.zero_grad()

        # Forward on FULL graph (safe)
        out = model(data.x, data.edge_index)

        loss = F.cross_entropy(out[batch_nodes], data.y[batch_nodes])

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / (len(perm) // batch_size + 1)

# Training loop
for epoch in range(21):
    loss = train()
    print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 0.0100
Epoch 1, Loss: 0.0095
Epoch 2, Loss: 0.0086
Epoch 3, Loss: 0.0077
Epoch 4, Loss: 0.0065
Epoch 5, Loss: 0.0061
Epoch 6, Loss: 0.0063
Epoch 7, Loss: 0.0052
Epoch 8, Loss: 0.0047
Epoch 9, Loss: 0.0045
Epoch 10, Loss: 0.0041
Epoch 11, Loss: 0.0039
Epoch 12, Loss: 0.0034
Epoch 13, Loss: 0.0031
Epoch 14, Loss: 0.0029
Epoch 15, Loss: 0.0026
Epoch 16, Loss: 0.0024
Epoch 17, Loss: 0.0023
Epoch 18, Loss: 0.0021
Epoch 19, Loss: 0.0020
Epoch 20, Loss: 0.0019


In [21]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def f1_auc_test(y_true, logits):
    y_true = y_true.cpu().numpy()
    proba = torch.softmax(logits, dim=1).cpu().numpy()
    pred = logits.argmax(dim=1).cpu().numpy()
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)
    c = proba.shape[1]
    if c < 2:
        return f1, float("nan")
    try:
        if c == 2:
            auc = roc_auc_score(y_true, proba[:, 1])
        else:
            auc = roc_auc_score(
                y_true, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        auc = float("nan")
    return f1, auc


@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())

    return acc, pred, out

acc, pred, gnn_output = test()
f1, auc = f1_auc_test(data.y[data.test_mask], gnn_output[data.test_mask])
print(f"Test Accuracy: {acc:.4f} | F1 (macro): {f1:.4f} | AUC (macro OVR): {auc:.4f}")


Test Accuracy: 0.9306 | F1 (macro): 0.9278 | AUC (macro OVR): 0.9880


In [22]:
print("Unique predictions:", pred.unique())
print("Unique labels:", data.y.unique())

Unique predictions: tensor([0, 1, 2, 3], device='cuda:0')
Unique labels: tensor([0, 1, 2, 3], device='cuda:0')
